# Day 7 — RFM Customer Analysis

This notebook builds customer-level Recency, Frequency and Monetary metrics from the cleaned positive-sales view. The dataset is retrieved from the documented UCI Online Retail source at runtime; numerical outputs are intentionally not hard-coded.

**Methodology**
- **Recency:** days since each customer's most recent positive purchase.
- **Frequency:** distinct invoice count, treating an invoice as an order.
- **Monetary:** total revenue from positive-sales transactions.
- **Reference date:** one day after the latest transaction retained for RFM, so the most recent purchaser has recency 1 rather than 0.
- **Scores:** relative quartile scores; higher is better. Recency is reversed because lower recency is better. These scores are comparative rankings, not absolute customer-value labels.

Cancellations/non-positive transactions are not silently rewritten; the RFM analytical view explicitly uses quantity > 0 and unit price > 0, consistent with the cleaning workflow. Customers without a usable CustomerID are excluded because RFM is customer-level.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from ucimlrepo import fetch_ucirepo

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.eda import positive_sales_view
from src.rfm_analysis import build_rfm, add_rfm_scores

online_retail = fetch_ucirepo(id=352)
raw = online_retail.data.features.copy()
raw.columns = [c.strip().lower().replace(' ', '_') for c in raw.columns]
raw['invoice_date'] = pd.to_datetime(raw['invoice_date'], errors='coerce')
for col in ['quantity', 'unit_price', 'customer_id']:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')
clean = raw.drop_duplicates().copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
sales = positive_sales_view(clean)
sales.shape

In [ ]:
rfm = build_rfm(sales)
reference_date = sales.loc[sales['customer_id'].notna(), 'invoice_date'].max() + pd.Timedelta(days=1)
print(f'Reference date: {reference_date.date()}')
print(f'Customers in RFM: {len(rfm):,}')
rfm.head()

## Distribution checks

RFM variables are commonly uneven: frequency and monetary value can have long right tails, while recency can be concentrated near recent dates. We inspect the actual distributions before deciding whether transformations are useful. The core RFM metrics remain in their original business units; transformations, if explored, are diagnostic rather than silently replacing the metrics.

In [ ]:
distribution = rfm[['recency', 'frequency', 'monetary']].describe().T
distribution['skewness'] = rfm[['recency', 'frequency', 'monetary']].skew()
distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    ax.hist(rfm[col].dropna(), bins=40)
    ax.set_title(f'{col.title()} distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Customers')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    ax.boxplot(rfm[col].dropna(), vert=True)
    ax.set_title(f'{col.title()} boxplot')
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

## RFM scoring

To make customers comparable, each metric is ranked into four equal-sized groups. Frequency and monetary receive 1–4 from low to high. Recency receives 4–1 from low to high because a smaller number of days is better. Ranking before `qcut` prevents tied values from collapsing quantile boundaries. The resulting three-digit code is a compact relative score, not a predictive model.

In [ ]:
rfm_scored = add_rfm_scores(rfm, n_bins=4)
rfm_scored[['r_score', 'f_score', 'm_score']].agg(['min', 'max', 'mean'])

rfm_scored.head()

In [ ]:
score_counts = (
    rfm_scored['rfm_score']
    .value_counts()
    .rename_axis('rfm_score')
    .reset_index(name='customers')
)
score_counts.head(15)

## RFM relationships

The following scatter plots show whether customers with stronger frequency and monetary value also tend to have more recent activity. No causal interpretation is implied.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(rfm['frequency'], rfm['monetary'], alpha=0.4)
axes[0].set_xlabel('Frequency (distinct invoices)')
axes[0].set_ylabel('Monetary (revenue)')
axes[0].set_title('Frequency vs Monetary')
axes[1].scatter(rfm['recency'], rfm['monetary'], alpha=0.4)
axes[1].set_xlabel('Recency (days)')
axes[1].set_ylabel('Monetary (revenue)')
axes[1].set_title('Recency vs Monetary')
plt.tight_layout()
plt.show()